# 00 Data Validation
Objectives:
- Validate the structural integrity of all core Olist tables (orders, items, sellers, customers, reviews, payments, products).
- Check primary and foreign key consistency between tables to ensure reliable joins.
- Verify time coverage and basic temporal logic (e.g., delivery dates after purchase dates).
- Define and validate SLA-related fields (delay_days, SLA violation, severe violation) at the order level.
- Assess availability and coverage of customer experience signals (reviews) for downstream analysis.
- Construct and export a clean `orders_sellers` dataset as the single source of truth for seller-level analysis.

Notes:
- All time-related columns are parsed as datetimes and checked for logical consistency before use.
- Multi-seller orders are examined, and a primary seller is assigned per order to enable seller-level SLA analysis.
- SLA delay is defined as `delay_days = delivered_customer_date - estimated_delivery_date`, with additional flags for SLA and severe violations.
- The final output is the cleaned `orders_sellers` table, will be used in later analysis.

## Validation Procedures

### 0. Import & Settings

In [22]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option('display.max_columns', 50)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

DATA_PROCEEDED = '../data/processed/'

DataTransformerRegistry.enable('default')

### 1. Load Datasets
All the datasets except for `geolocation` are loaded. All time related columns are parsed as datetime objects.

In [23]:
from data.load_raw import (
    load_orders,
    load_items,
    load_reviews,
    load_sellers,
    load_customers,
    load_payments,
    load_products,
)

orders = load_orders()
items = load_items()
reviews = load_reviews()
sellers = load_sellers()
customers = load_customers()
payments = load_payments()
products = load_products()


### 2. Table Overviews
We display fundamental statistics of each table to understand their structure and content.

In [24]:
# All the tables are loaded except geolocation
from utils.eda import quick_overview

quick_overview(orders, "orders")
quick_overview(items, "order_items")
quick_overview(reviews, "order_reviews")
quick_overview(sellers, "sellers")
quick_overview(customers, "customers")
quick_overview(payments, "order_payments")
quick_overview(products, "products")

=== orders ===
(99441, 8)
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
order_delivered_customer_date    0.029817
order_delivered_carrier_date     0.017930
order_approved_at                0.001609
order_id                         0.000000
order_purchase_timestamp         0.000000
order_status                     0.000000
customer_id                      0.000000
order_estimated_delivery_date    0.000000
dtype: float64

=== order_items ===
(112650, 7)
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
pr

### 3. Data Integrity Checks

#### 3.1 Primary key integrity checks
Number of unique values in primary key columns are compared against the total number of rows to ensure data integrity.

In [25]:
from validation.data_quality import validate_primary_keys


pk_ok = validate_primary_keys({
    "orders": (orders, ["order_id"]),
    "customer": (customers, ["customer_id"]),
    "sellers": (sellers, ["seller_id"]),
    "products": (products, ["product_id"]),
    "order_items": (items, ["order_id", "order_item_id"]),
})

[PK PASSED] orders: 99441 rows, 99441 unique rows by ['order_id']
[PK PASSED] customer: 99441 rows, 99441 unique rows by ['customer_id']
[PK PASSED] sellers: 3095 rows, 3095 unique rows by ['seller_id']
[PK PASSED] products: 32951 rows, 32951 unique rows by ['product_id']
[PK PASSED] order_items: 112650 rows, 112650 unique rows by ['order_id', 'order_item_id']

Overall PK check: ALL PASSED


#### 3.2 Foreign key integrity checks
Make sure the foreign key columns only contain values that exist in the referenced primary key columns.

In [26]:
from validation.data_quality import validate_foreign_keys

fk_ok = validate_foreign_keys([
    ("order_items.order_id with orders.order_id", items, "order_id", orders, "order_id"),
    ("order_items.seller_id with sellers.seller_id", items, "seller_id", sellers, "seller_id"),
    ("order_reviews.order_id with orders.order_id", reviews, "order_id", orders, "order_id"),
    ("order_payments.order_id with orders.order_id", payments, "order_id", orders, "order_id"),
])

[FK PASSED]: 100.0000% of non-null order_id values found in order_id (112650/112650)
[FK PASSED]: 100.0000% of non-null seller_id values found in seller_id (112650/112650)
[FK PASSED]: 100.0000% of non-null order_id values found in order_id (99224/99224)
[FK PASSED]: 100.0000% of non-null order_id values found in order_id (103886/103886)

Overall FK check: ALL PASSED


### 4. Time Coverage Validation
#### 4.1 Check the time range of the datasets. 
Check if all the delivery date are later than the order date to ensure the data consistency.

In [27]:
# Time Range Checks
orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

# time logic checks
from validation.data_quality import validate_time_logic
validate_time_logic(orders, "order_purchase_timestamp", "order_delivered_customer_date")

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

[PASSED]: All records have order_delivered_customer_date >= order_purchase_timestamp


True

#### 4.2 Check the time coverage of `Orders`, `Order Items`, and `Reviews`

In [28]:
from utils.eda import time_coverage
time_coverage(orders, "Orders", "order_purchase_timestamp")
time_coverage(items, "Order Items", "shipping_limit_date")
time_coverage(reviews, "Reviews", "review_creation_date")

Orders
  Start: 2016-09-04 21:15:19
  End:   2018-10-17 17:30:18
  Duration: 772 days

Order Items
  Start: 2016-09-19 00:15:34
  End:   2020-04-09 22:35:08
  Duration: 1298 days

Reviews
  Start: 2016-10-02 00:00:00
  End:   2018-08-31 00:00:00
  Duration: 698 days



#### 4.3 Investigate on numbers of orders for each month

In [29]:
monthly_orders = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
print("Monthly order counts:")
print(monthly_orders)

Monthly order counts:
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64


### 5. Seller-Order Linkage Verification

#### 5.1 Investigate on the proportion of orders with multiple sellers

In [30]:
# Multi-seller order analysis
sellers_per_order = items.groupby('order_id')['seller_id'].nunique()
print(sellers_per_order.value_counts().sort_index())
print(f"\nPercent of orders with multiple sellers: {(sellers_per_order > 1).mean():.2%}")

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

Percent of orders with multiple sellers: 1.30%


#### 5.2 Investigate the statistics of the number of orders per seller.

In [31]:
# Seller activity statistics
orders_per_seller = items.groupby('seller_id')['order_id'].nunique()
print(orders_per_seller.describe())

count    3095.000000
mean       32.313409
std       105.139763
min         1.000000
25%         2.000000
50%         6.000000
75%        21.500000
max      1854.000000
Name: order_id, dtype: float64


### 6. SLA Calculation & Quality Checks

#### 6.1 Proportion of delivered orders

In [32]:
# how much of the orders are delayed
delivered = orders["order_delivered_customer_date"].notna()


print("=== Delivery Status Overview ===")
print(f"Total orders: {len(orders):,}")
print(f"Delivered orders: {delivered.sum():,} ({delivered.mean():.2%})")
print(f"Not delivered: {(~delivered).sum():,} ({(~delivered).mean():.2%})")

=== Delivery Status Overview ===
Total orders: 99,441
Delivered orders: 96,476 (97.02%)
Not delivered: 2,965 (2.98%)


#### 6.2 Delivery delay statistics
Only delivered orders are considered in this section

In [33]:
from features.sla_metrics import calculate_delay_days
from utils.eda import show_delay_bucket

# culculate delay days
orders = calculate_delay_days(orders)

# see the delay data
delay_data = orders.loc[delivered, "delay_days"]

print("\n=== Delay Distribution (Delivered Orders) ===")
show_delay_bucket(delay_data)



=== Delay Distribution (Delivered Orders) ===


Delay >30 days: 345 (0.36%)
Delay 8-30 days: 2,518 (2.61%)
Delay 1-7 days: 3,672 (3.81%)
On-time or early delivery: 89,941 (93.23%)


#### 6.3 Calculate the proportion of orders in each SLA bucket

In [34]:
# culculated use the sla metrics functions defined in sla_metrics.py
from features.sla_metrics import add_sla_violation_flags
from features.sla_metrics import get_sla_summary

orders = add_sla_violation_flags(orders)
sla_summary = get_sla_summary(orders)
sla_summary


,Metric,Count,Rate (of delivered)
0,Orders with any delay (SLA violation),"6,535",6.77%
1,Orders with severe delay (>7 days),"2,863",2.97%


Visualization of the proportion of orders in each SLA bucket

In [35]:
df_delay = orders.loc[delivered, ["delay_days"]].copy()
chart = (
    alt.Chart(df_delay)
    .mark_bar()
    .encode(
        x=alt.X(
            "delay_days:Q",
            bin=alt.Bin(maxbins=50),
            title="Delay (days)"
        ),
        y=alt.Y(
            "count():Q",
            title="Num of orders"
        ),
        tooltip=[
            alt.Tooltip("delay_days:Q", bin=True, title="Delay range (days)"),
            alt.Tooltip("count():Q", title="Order count"),
        ],
    )
    .properties(
        width=500,
        height=350,
        title="Distribution of Delivery Delays"
    )
    .configure_view(
        strokeWidth=0
    )
    .configure_axis(
        labelFontSize=11,
        titleFontSize=12
    )
)

chart

alt.Chart(...)

#### 6.4 Extreme Delay Outliers & `has_time_anomaly`
Document known data quality issues in delay distribution before downstream analysis uses this data.

In [44]:
# Extreme delay outliers (> 60 days)
extreme_delay = orders[delivered & (orders["delay_days"] > 60)]
print(f"Orders with delay > 60 days: {len(extreme_delay):,}  ({len(extreme_delay)/delivered.sum():.2%} of delivered)")
print(extreme_delay[["order_id", "delay_days", "order_status"]].head(10))

# has_time_anomaly: delivered_date < purchase_date
time_anomaly_mask = (
    orders["order_delivered_customer_date"].notna() &
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"])
)
print(f"\nTime anomalies (delivered < purchased): {time_anomaly_mask.sum():,}")

Orders with delay > 60 days: 79  (0.08% of delivered)
                               order_id  delay_days order_status
1621   a4efaffc506a395c9cea7402b078c1e5        73.0    delivered
3077   8b7fd198ad184563c231653673e75a7f        91.0    delivered
3202   4f39a94d6e474819d898d6df7d394996       112.0    delivered
4666   b31c7dea63bb08f8cdd1ec32514ccf0b       109.0    delivered
4846   f390346ecbf61a6b81b92017a7410df1        63.0    delivered
5335   c679466a1ba6a8bdedcb2396f9e1dcbd        66.0    delivered
7740   00d1289d5125017a90e528b5a7cee91f        74.0    delivered
11399  47b40429ed8cce3aee9199792275433f       175.0    delivered
12813  c2a550cc5f966506b717532441c221e5       104.0    delivered
13672  d8dbb44d7c5b1fd8e7f41b49e27053d7        96.0    delivered

Time anomalies (delivered < purchased): 0


#### 6.5 Payments / GMV Validation
Validate the `payments` table and compute order-level GMV that will be merged into `orders_sellers`.
This is required for all downstream ROI and cost-of-violation calculations.

In [37]:
# Basic payments table validation
print("=== Payments Table ===")
print(f"Rows:          {len(payments):,}")
print(f"Unique orders: {payments['order_id'].nunique():,}")
print(f"\nPayment value distribution (per payment row):")
print(payments["payment_value"].describe().round(2))
print(f"\nZero-value payments: {(payments['payment_value'] == 0).sum():,}")

# Aggregate to order level
# Some orders have installments or multiple payment methods — sum all rows per order
order_gmv = (
    payments.groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_gmv"})
)

print(f"\n=== Order-level GMV (aggregated) ===")
print(order_gmv["order_gmv"].describe().round(2))

# GMV coverage check against orders table
gmv_coverage = order_gmv["order_id"].isin(orders["order_id"]).mean()
print(f"\nGMV coverage: {gmv_coverage:.2%} of payment order_ids found in orders table")
print(f"Total platform GMV: BRL {order_gmv['order_gmv'].sum():,.0f}")

=== Payments Table ===
Rows:          103,886
Unique orders: 99,440

Payment value distribution (per payment row):
count    103886.00
mean        154.10
std         217.49
min           0.00
25%          56.79
50%         100.00
75%         171.84
max       13664.08
Name: payment_value, dtype: float64

Zero-value payments: 9

=== Order-level GMV (aggregated) ===
count    99440.00
mean       160.99
std        221.95
min          0.00
25%         62.01
50%        105.29
75%        176.97
max      13664.08
Name: order_gmv, dtype: float64

GMV coverage: 100.00% of payment order_ids found in orders table
Total platform GMV: BRL 16,008,872


### 7. Customer experience signal validation
The valid review rate is culculated by `delivered orders that has at least 1 review` / `number of orders delivered`
Minimum acceptable valid review rate is 90%.

In [38]:
from validation.data_quality import validate_review_coverage

R1 = validate_review_coverage(orders, reviews)

[REVIEW_COVERAGE PASSED] Delivered orders with valid review score: 99.33% (min required: 90.00%)


#### 7.2 Review Score Distribution
Understand the shape of the review score distribution before using it as a CX outcome metric.
Note: Olist reviews show a J-curve (1-star and 5-star dominate) — this makes mean_review sensitive to skew,
which is why non-parametric tests (Mann-Whitney U) are preferred in downstream analysis.

In [39]:
review_score_counts = (
    reviews["review_score"]
    .value_counts()
    .reset_index()
    .rename(columns={"review_score": "score", "count": "n"})
    .sort_values("score")
)

review_dist_chart = (
    alt.Chart(review_score_counts)
    .mark_bar()
    .encode(
        x=alt.X("score:O", title="Review Score (1–5)", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("n:Q", title="Number of Reviews"),
        color=alt.condition(
            alt.datum.score <= 3,
            alt.value("#e45756"),   # red for low ratings (≤3)
            alt.value("#4c78a8"),   # blue for high ratings
        ),
        tooltip=[
            alt.Tooltip("score:O", title="Score"),
            alt.Tooltip("n:Q", title="Count", format=","),
        ],
    )
    .properties(
        title="Review Score Distribution — J-curve: 1-star and 5-star dominate",
        width=380,
        height=260,
    )
)

pct = reviews["review_score"].value_counts(normalize=True).sort_index() * 100
for s, p in pct.items():
    print(f"  Score {s}: {p:.1f}%")

review_dist_chart

  Score 1: 11.5%
  Score 2: 3.2%
  Score 3: 8.2%
  Score 4: 19.3%
  Score 5: 57.8%


alt.Chart(...)

### 8. Create orders_sellers Table for Future Analysis

In [40]:
from validation.data_quality import validate_missing_sellers, validate_missing_by_column
from data.preprocessing import select_primary_seller, build_orders_sellers

# Use order_item_id == 1 as the primary seller for each order
items_main = select_primary_seller(items, primary_item_id=1)

print("=== Seller Assignment ===")
print(f"Total orders in orders table: {len(orders):,}")
print(f"Orders with item #1 entry in items: {len(items_main):,}")
print(f"Match rate: {len(items_main) / len(orders):.2%}")


=== Seller Assignment ===
Total orders in orders table: 99,441
Orders with item #1 entry in items: 98,666
Match rate: 99.22%


In [41]:
# 8.2 Merge to create integrated order–seller dataset
cols_keep = [
    "order_id",
    "seller_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delay_days",
    "is_sla_violation",
    "is_severe_violation",
    "has_time_anomaly",
    "product_category_name_english",
]

orders_sellers = build_orders_sellers(
    orders=orders,
    items=items,
    products=products,
    primary_item_id=1,
    cols_keep=cols_keep,
)

# Merge order-level GMV from payments table
# order_gmv was computed in Section 6.5 (sum of payment_value per order_id)
orders_sellers = orders_sellers.merge(order_gmv, on="order_id", how="left")

print("\n=== Final Dataset Quality ===")
print(f"Total rows:    {len(orders_sellers):,}")
print(f"Columns:       {len(orders_sellers.columns)}")
print(f"GMV missing:   {orders_sellers['order_gmv'].isna().sum():,} orders ({orders_sellers['order_gmv'].isna().mean():.2%})")

# validate seller_id missing rate (against DATA_QUALITY_THRESHOLDS["max_missing_seller"])
validate_missing_sellers(orders_sellers)

# print missing rate for each column
validate_missing_by_column(orders_sellers, name="orders_sellers")


=== Final Dataset Quality ===
Total rows:    99,441
Columns:       13
GMV missing:   1 orders (0.00%)

=== Missing Seller Assignment Overview ===
Orders without seller assignment: 775 (0.7794%)

Missing seller by order_status:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

=== Missing values per column: orders_sellers ===
  order_id: 0.00%
  seller_id: 0.78%
  customer_id: 0.00%
  order_status: 0.00%
  order_purchase_timestamp: 0.00%
  order_delivered_customer_date: 2.98%
  order_estimated_delivery_date: 0.00%
  delay_days: 2.98%
  is_sla_violation: 0.00%
  is_severe_violation: 0.00%
  has_time_anomaly: 0.00%
  product_category_name_english: 2.22%
  order_gmv: 0.00%


In [42]:
# export the cleaned dataset
output_path = DATA_PROCEEDED + "orders_sellers.csv"

orders_sellers.to_csv(output_path, index=False)

print("\n=== Export Summary ===")
print(f"File size: {len(orders_sellers):,} rows x {len(orders_sellers.columns)} columns")
print(f"Dataset exported to: {output_path}")



=== Export Summary ===
File size: 99,441 rows x 13 columns
Dataset exported to: ../data/processed/orders_sellers.csv


## Validation Summary

In [43]:
summary_metrics = {}

# variable counts
summary_metrics["n_orders"] = len(orders)
summary_metrics["n_sellers"] = orders_sellers["seller_id"].nunique()
summary_metrics["n_customers"] = orders_sellers["customer_id"].nunique()

# Time Range Checks
summary_metrics["time_range_start"] = str(orders["order_purchase_timestamp"].min().date())
summary_metrics["time_range_end"] = str(orders["order_purchase_timestamp"].max().date())

# delivery status
delivered_mask = orders["order_delivered_customer_date"].notna()
summary_metrics["delivered_rate"] = delivered_mask.mean()

# multi-seller order rate
sellers_per_order = items.groupby("order_id")["seller_id"].nunique()
summary_metrics["multi_seller_rate"] = (sellers_per_order > 1).mean()

# SLA violation rates
sla_mask = orders_sellers["is_sla_violation"]
severe_mask = orders_sellers["is_severe_violation"]
summary_metrics["sla_violation_rate"] = sla_mask.mean()
summary_metrics["severe_violation_rate"] = severe_mask.mean()

# missing seller assignment rate
missing_seller_mask = orders_sellers["seller_id"].isna()
summary_metrics["missing_seller_rate"] = missing_seller_mask.mean()

# customer review rate
valid_review_rate = R1
summary_metrics["valid_review_rate"] = valid_review_rate

# GMV metrics
summary_metrics["total_gmv_brl"] = orders_sellers["order_gmv"].sum()
summary_metrics["avg_order_gmv_brl"] = orders_sellers["order_gmv"].mean()
summary_metrics["gmv_at_risk_brl"] = orders_sellers.loc[sla_mask, "order_gmv"].sum()
summary_metrics["gmv_at_risk_pct"] = (
    summary_metrics["gmv_at_risk_brl"] / summary_metrics["total_gmv_brl"]
)

summary_metrics

{'n_orders': 99441,
 'n_sellers': 3088,
 'n_customers': 99441,
 'time_range_start': '2016-09-04',
 'time_range_end': '2018-10-17',
 'delivered_rate': np.float64(0.9701833247855512),
 'multi_seller_rate': np.float64(0.01295279022155555),
 'sla_violation_rate': np.float64(0.06571736004263835),
 'severe_violation_rate': np.float64(0.02879094136221478),
 'missing_seller_rate': np.float64(0.007793566034130791),
 'valid_review_rate': 0.9933041729720765,
 'total_gmv_brl': np.float64(16008872.120000001),
 'avg_order_gmv_brl': np.float64(160.9902666934835),
 'gmv_at_risk_brl': np.float64(1150909.75),
 'gmv_at_risk_pct': np.float64(0.07189199472473518)}

## Executive Summary - Data Validation & SLA Definition

- **Dataset coverage**
  - Total orders: **99,441**, customers: **99,441**, sellers: **3,088**.
  - The dataset covers orders from **2016-09-04** to **2018-10-17**, with no major gaps in monthly order volume.
  - Delivered orders account for **97.02%** of all orders.

- **Order–seller structure**
  - **98.7%** of orders involve a single seller.
  - Multi-seller orders are rare at **1.3%**, so assigning a primary seller per order is a reasonable simplification for seller-level SLA analysis.

- **SLA definition & delay behaviour**
  - SLA delay is defined as  
    `delay_days = delivered_customer_date - estimated_delivery_date`, computed only for orders with both timestamps present.
  - Among delivered orders, **6.57%** violate the promised delivery date (`delay_days > 0`).
  - **2.88%** of delivered orders experience **severe delay** (`delay_days > 7`), which will be used as "high customer harm" events in downstream analysis and simulations.
  - Extreme outliers (delay > 60 days) are a tiny fraction of delivered orders and do not meaningfully affect aggregate SLA statistics.
  - `has_time_anomaly=True` flags the rare orders where `delivered_date < purchased_date` (logical impossibility); these are retained but excluded from SLA delay calculations.

- **GMV coverage**
  - Order-level GMV is aggregated from the payments table and merged into `orders_sellers`.
  - GMV-at-risk (orders with any SLA violation) is reported in `summary_metrics` and used in downstream ROI simulations.

- **Customer experience signal availability**
  - Among delivered orders, **99.33%** have a valid review score.
  - Review score distribution is J-shaped (1-star and 5-star dominate); this makes `mean_review` sensitive to skew, which is why non-parametric tests are preferred in downstream analysis.
  - This provides a sufficiently large sample to study the relationship between SLA reliability, review scores, and repeat behaviour.

- **Data quality & joins**
  - `seller_id` is missing for only **0.78%** of orders, and these will be excluded from seller-level risk analysis.
  - No major structural issues were found in key joins between orders, items, sellers, reviews, and payments.

These checks confirm that the Olist dataset is structurally sound and suitable for:
1. Seller-level risk analysis,
2. Customer impact validation,
3. GMV-at-risk quantification, and
4. ROI-oriented intervention simulation in subsequent notebooks.